In [1]:
import os
import librosa
import torch
import numpy as np
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import pandas as pd

In [2]:
# Thư mục gốc chứa dữ liệu
input_dir = r"nuhive_processed"
logmel_dir = r"logmel_spectrograms"
stft_dir = r"stft_spectrograms"
os.makedirs(logmel_dir, exist_ok = True)
os.makedirs(stft_dir, exist_ok = True)

# Tham số spectrogram
sample_rate = 22050
n_fft = 2048
n_mels = 128
hop_length = 512

In [5]:
print("🔹 Bắt đầu trích xuất Log-Mel spectrograms...")
for label in os.listdir(input_dir):
    label_dir = os.path.join(input_dir, label)
    if not os.path.isdir(label_dir):
        continue

    files = [f for f in os.listdir(label_dir) if f.endswith(".wav")]
    for i, file in enumerate(tqdm(files, desc=f"Label: {label}")):
        file_path = os.path.join(label_dir, file)
        try:
            #Log-Mel
            y, sr = librosa.load(file_path, sr=sample_rate)
            mel = librosa.feature.melspectrogram(
                y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
            log_mel = np.log(np.sqrt(mel + 1e-9) + 1e-9)
            logmel_tensor = torch.tensor(log_mel).unsqueeze(0)
            torch.save(logmel_tensor, os.path.join(logmel_dir, f"{label}_{i}.pt"))

            #STFT
            stft = librosa.stft(y=y, n_fft=n_fft, hop_length=hop_length)
            stft_mag = np.abs(stft) 
            stft_tensor = torch.tensor(stft_mag).unsqueeze(0)
            torch.save(stft_tensor, os.path.join(stft_dir, f"{label}_{i}.pt"))

        except Exception as e:
            print(f"Lỗi xử lý {file}: {e}")

print("✅ Hoàn thành lưu spectrograms!")


🔹 Bắt đầu trích xuất Log-Mel spectrograms...


Label: bee:   0%|          | 0/5473 [00:00<?, ?it/s]

Label: noqueen: 100%|██████████| 4861/4861 [01:24<00:00, 57.81it/s]

✅ Hoàn thành lưu spectrograms!


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained models
vgg16 = nn.Sequential(*list(models.vgg16(weights="IMAGENET1K_V1").features.children())[:3]).to(device).eval()
resnet50 = models.resnet50(weights="IMAGENET1K_V1").to(device).eval()
resnet50 = nn.Sequential(
    resnet50.conv1,
    resnet50.bn1,
    resnet50.relu,
    resnet50.maxpool,
    resnet50.layer1,
    resnet50.layer2 
).to(device).eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))

def extract_deep_features(tensor):
    tensor = tensor.repeat(3, 1, 1)  # (3, 128, T)
    tensor = torch.nn.functional.interpolate(
        tensor.unsqueeze(0), size=(224, 224),
        mode='bilinear', align_corners=False).squeeze(0)
    tensor = transform(tensor).unsqueeze(0).to(device)

    with torch.no_grad():
        feat_vgg = vgg16(tensor)                       # 
        feat_vgg = global_avg_pool(feat_vgg)           #
        feat_vgg = feat_vgg.view(-1)                   # (256,)

        feat_res = resnet50(tensor)                    # (1, 256, 56, 56)
        feat_res = global_avg_pool(feat_res)           # (1, 256, 1, 1)
        feat_res = feat_res.view(-1)                   # (256,)

    return torch.cat([feat_vgg, feat_res], dim=0).cpu().numpy()  


In [5]:
X_logmel, X_stft, X, y = [], [], [], []
label_map = {"bee": 0, "nobee": 1, "noqueen": 2}

print("🔹 Trích xuất đặc trưng từ VGG16 + ResNet50...")
print(f"Log-Mel")
for file in tqdm(os.listdir(logmel_dir)):
    if not file.endswith(".pt"): continue
    label = file.split("_")[0]
    tensor = torch.load(os.path.join(logmel_dir, file), weights_only = False)
    feat = extract_deep_features(tensor)
    X_logmel.append(feat)
    y.append(label_map[label])
print(f"STFT")
for file in tqdm(os.listdir(stft_dir)):
    if not file.endswith(".pt"): continue
    label = file.split("_")[0]
    tensor = torch.load(os.path.join(stft_dir, file), weights_only = False)
    feat = extract_deep_features(tensor)
    X_stft.append(feat)

X_logmel = np.array(X_logmel)
X_stft = np.array(X_stft)

for v1, v2 in zip(X_logmel, X_stft):
    combined_vec = np.concatenate([v1, v2])
    X.append(combined_vec)

X = np.array(X)
y = np.array(y)
print(f"✅ Dataset hoàn chỉnh: X={X.shape}, y={y.shape}")


🔹 Trích xuất đặc trưng từ VGG16 + ResNet50...
Log-Mel


  0%|          | 0/13792 [00:00<?, ?it/s]

100%|██████████| 13792/13792 [25:46<00:00,  8.92it/s] 


STFT


100%|██████████| 13792/13792 [22:51<00:00, 10.06it/s]


✅ Dataset hoàn chỉnh: X=(13792, 1152), y=(13792,)


In [6]:
models_ml = {
    "Logistic Regression": LogisticRegression(max_iter=1500),
    "SVM (RBF)": SVC(kernel='rbf', gamma='scale', C=10),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, criterion = 'log_loss'),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.01,
                             subsample=0.8, colsample_bytree=0.8,
                             random_state=42, use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(n_estimators=200, learning_rate=0.01, random_state=42)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models_ml.items():
    print(f"⚙️ Huấn luyện mô hình: {name}")
    accs, f1s, recs, precs = [], [], [], []

    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)

        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        accs.append(accuracy_score(y_val, y_pred))
        f1s.append(f1_score(y_val, y_pred, average='macro'))
        recs.append(recall_score(y_val, y_pred, average='macro'))
        precs.append(precision_score(y_val, y_pred, average='macro'))

    results[name] = {
        "Accuracy": np.mean(accs),
        "F1": np.mean(f1s),
        "Recall": np.mean(recs),
        "Precision": np.mean(precs)
    }
    print(f"✅ {name}: F1={np.mean(f1s):.4f}, Acc={np.mean(accs):.4f}")


⚙️ Huấn luyện mô hình: Logistic Regression
✅ Logistic Regression: F1=0.7849, Acc=0.8140
⚙️ Huấn luyện mô hình: SVM (RBF)
✅ SVM (RBF): F1=0.8206, Acc=0.8516
⚙️ Huấn luyện mô hình: KNN
✅ KNN: F1=0.8057, Acc=0.8384
⚙️ Huấn luyện mô hình: Random Forest
✅ Random Forest: F1=0.8113, Acc=0.8463
⚙️ Huấn luyện mô hình: XGBoost


C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [09:58:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [09:59:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [09:59:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [10:00:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:7

✅ XGBoost: F1=0.7997, Acc=0.8370
⚙️ Huấn luyện mô hình: LightGBM
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.058190 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 292470
[LightGBM] [Info] Number of data points in the train set: 11033, number of used features: 1151
[LightGBM] [Info] Start training from score -0.924299
[LightGBM] [Info] Start training from score -1.383489
[LightGBM] [Info] Start training from score -1.042739


C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.053799 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 292461
[LightGBM] [Info] Number of data points in the train set: 11033, number of used features: 1150
[LightGBM] [Info] Start training from score -0.924299
[LightGBM] [Info] Start training from score -1.383127
[LightGBM] [Info] Start training from score -1.042996


...

C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.056944 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 292471
[LightGBM] [Info] Number of data points in the train set: 11034, number of used features: 1151
[LightGBM] [Info] Start training from score -0.924161
[LightGBM] [Info] Start training from score -1.383579
[LightGBM] [Info] Start training from score -1.042829


C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.056345 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 292462
[LightGBM] [Info] Number of data points in the train set: 11034, number of used features: 1150
[LightGBM] [Info] Start training from score -0.924161
[LightGBM] [Info] Start training from score -1.383579
[LightGBM] [Info] Start training from score -1.042829
✅ LightGBM: F1=0.8041, Acc=0.8389


C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [7]:
tree_models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, criterion = 'log_loss'),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.01,
                             subsample=0.8, colsample_bytree=0.8,
                             random_state=42, use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(n_estimators=200, learning_rate=0.01, random_state=42)
}

In [8]:
feature_scores = pd.DataFrame()

for name, model in tree_models.items():
    print(f"⚙️ Huấn luyện mô hình: {name}")
    fold = int(0)
    for train_idx, val_idx in skf.split(X, y):

        fold+=1

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)

        model.fit(X_train, y_train)
        importances = model.feature_importances_
        col = name + str(fold)
        feature_scores[col] = importances
    

⚙️ Huấn luyện mô hình: Random Forest
⚙️ Huấn luyện mô hình: XGBoost


C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [10:26:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [10:27:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [10:27:56] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [10:28:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:7

⚙️ Huấn luyện mô hình: LightGBM
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.051959 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 292470
[LightGBM] [Info] Number of data points in the train set: 11033, number of used features: 1151
[LightGBM] [Info] Start training from score -0.924299
[LightGBM] [Info] Start training from score -1.383489
[LightGBM] [Info] Start training from score -1.042739
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.060995 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 292461
[LightGBM] [Info] Number of data points in the train set: 11033, number of used features: 1150
[LightGBM] [Info] Start training from score -0.924299
[LightGBM] [Info] Start training from score -1.383127
[LightGBM] [Info] Start training from score -1.042996
[LightGBM] [Info] Auto-choosing col-wise multi-threading, 

In [15]:
final_fs = pd.DataFrame({
    r"Random Forest": np.mean(np.array([feature_scores['Random Forest1'], feature_scores['Random Forest2'], feature_scores['Random Forest3'], feature_scores['Random Forest4'], feature_scores['Random Forest5']]), axis = 0),
    r"XGBoost": np.mean(np.array([feature_scores['XGBoost1'], feature_scores['XGBoost2'], feature_scores['XGBoost3'], feature_scores['XGBoost4'], feature_scores['XGBoost5']]), axis = 0),
    r"LightGBM": np.mean(np.array([feature_scores['LightGBM1'], feature_scores['LightGBM2'], feature_scores['LightGBM3'], feature_scores['LightGBM4'], feature_scores['LightGBM5']]), axis = 0)
})
final_fs

,Random Forest,XGBoost,LightGBM
0,0.000772,0.000192,0.2
1,0.000912,0.000342,7.0
2,0.003451,0.001067,83.4
3,0.000467,0.000193,0.0
4,0.000310,0.000372,3.0
...,...,...,...
1147,0.000473,0.000747,5.4
1148,0.000373,0.001006,20.8
1149,0.000456,0.000572,3.4
1150,0.000764,0.001217,47.6


In [67]:
rf_top_features = final_fs["Random Forest"].sort_values(ascending = False).head(150).index.tolist()
xgb_top_features = final_fs["XGBoost"].sort_values(ascending = False).head(150).index.tolist()
lgbm_top_features = final_fs["LightGBM"].sort_values(ascending = False).head(150).index.tolist()

In [68]:
top_features = {
    "Random Forest": np.array(rf_top_features),
    "XGBoost": np.array(xgb_top_features),
    "LightGBM": np.array(lgbm_top_features)
}

In [66]:
new_results = {}

for name, model in tree_models.items():
    print(f"⚙️ Huấn luyện mô hình: {name}")
    accs, f1s, recs, precs = [], [], [], []
    fold = 0
    for train_idx, val_idx in skf.split(X, y):
        fold += 1
        print(f"Fold {fold}")
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)

        new_X_train = X_train[:, top_features[name]]
        new_X_val = X_val[:, top_features[name]]

        model.fit(new_X_train, y_train)
        y_pred = model.predict(new_X_val)

        accs.append(accuracy_score(y_val, y_pred))
        f1s.append(f1_score(y_val, y_pred, average='macro'))
        recs.append(recall_score(y_val, y_pred, average='macro'))
        precs.append(precision_score(y_val, y_pred, average='macro'))

    new_results[name] = {
        "Accuracy": np.mean(accs),
        "F1": np.mean(f1s),
        "Recall": np.mean(recs),
        "Precision": np.mean(precs)
    }
    print(f"✅ {name}: F1={np.mean(f1s):.4f}, Acc={np.mean(accs):.4f}")


⚙️ Huấn luyện mô hình: Random Forest
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
✅ Random Forest: F1=0.8175, Acc=0.8496
⚙️ Huấn luyện mô hình: XGBoost
Fold 1


C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [11:37:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 2


C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [11:37:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 3


...

C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 4
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007621 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51000
[LightGBM] [Info] Number of data points in the train set: 11034, number of used features: 200
[LightGBM] [Info] Start training from score -0.924161
[LightGBM] [Info] Start training from score -1.383579
[LightGBM] [Info] Start training from score -1.042829


C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 5
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008328 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51000
[LightGBM] [Info] Number of data points in the train set: 11034, number of used features: 200
[LightGBM] [Info] Start training from score -0.924161
[LightGBM] [Info] Start training from score -1.383579
[LightGBM] [Info] Start training from score -1.042829
✅ LightGBM: F1=0.8052, Acc=0.8393


C:\Users\USER\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
